# TOBi Routing & Escalation — Executive Management Dashboard

Built directly on **BigQuery** and organised around the five business objectives.
Every panel runs a small **aggregate query** (GROUP BY) — it never pulls the full
session table into memory, so it scales to tens of millions of rows.

| Objective | Question it answers | Section |
|---|---|---|
| 1 | Where are technical issues incorrectly routed/escalated? | 2, 3 |
| 2 | What does misrouting cost (time / channel load / repeats)? | 4 |
| 3 | What drives the wrong routing (topic, entry point)? | 3, 5 |
| 4 | What should we change (routing logic / escalation)? | 6 |
| 5 | How do we track first-contact resolution & efficiency? | 7 |

**Prerequisite:** materialise `session_master` once from
`standalone/session_master_query.sql` (`CREATE OR REPLACE TABLE your_dataset.session_master AS <body>`),
then set the config below.

## 1. Setup & BigQuery connection

In [ ]:
# %pip install google-cloud-bigquery db-dtypes pandas matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from google.cloud import bigquery

RED='#E60000'; DARK='#25282A'; GREY='#7E8083'; LGREY='#E9EAEC'
GREEN='#009900'; AMBER='#FBA600'; BLUE='#0077C8'; INK='#4A4D4E'
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.edgecolor':LGREY,
  'axes.grid':True,'grid.color':LGREY,'axes.axisbelow':True,'axes.spines.top':False,
  'axes.spines.right':False,'axes.titlesize':13,'axes.titleweight':'bold','figure.dpi':110})

def vlabels(ax,bars,fmt='{:.0f}',horiz=False,pad=3):
    for b in bars:
        v=b.get_width() if horiz else b.get_height()
        xy=(v,b.get_y()+b.get_height()/2) if horiz else (b.get_x()+b.get_width()/2,v)
        off=(4,0) if horiz else (0,pad)
        ax.annotate(fmt.format(v),xy,xytext=off,textcoords='offset points',
                    va='center',ha=('left' if horiz else 'center'),fontweight='bold',fontsize=9)

In [ ]:
# ---- CONFIG ----
PROJECT = 'vf-pt-copsvertex-live'
DATASET = 'your_dataset'          # where you materialised session_master
TABLE   = 'session_master'
DATE_FROM, DATE_TO = '2024-01-01', '2025-12-31'   # exclude null/1900 junk dates

client = bigquery.Client(project=PROJECT)
M = f'`{PROJECT}.{DATASET}.{TABLE}`'
W = f"DATE(START_MOMENT) BETWEEN '{DATE_FROM}' AND '{DATE_TO}'"
def q(sql): return client.query(sql).to_dataframe()
print('source:', M)

## 2. Headline KPIs

In [ ]:
k = q(f'''
SELECT COUNTIF(is_technical_topic) tech,
       COUNTIF(is_technical_topic AND is_bot_contained) contained,
       COUNTIF(is_correct_technical_route) correct,
       COUNTIF(is_hard_misroute) hard,
       COUNTIF(is_soft_misroute) soft,
       SUM(IF(is_hard_misroute OR is_soft_misroute, n_transfers, 0)) extra_handovers
FROM {M} WHERE {W}''').iloc[0]

tech=int(k.tech)
cards=[('Technical sessions',f'{tech/1e6:.1f}M','in window',INK),
       ('Misrouted',f'{100*(k.hard+k.soft)/tech:.0f}%',f'hard {100*k.hard/tech:.0f}% + soft {100*k.soft/tech:.0f}%',RED),
       ('Bot-contained (FCR)',f'{100*k.contained/tech:.0f}%','resolved first contact',GREEN),
       ('Extra handovers',f'{k.extra_handovers/1e3:.0f}k','caused by misroutes',BLUE)]
fig,ax=plt.subplots(1,4,figsize=(15,2.5))
for a,(t,v,s,c) in zip(ax,cards):
    a.axis('off')
    a.add_patch(FancyBboxPatch((.04,.08),.92,.84,boxstyle='round,pad=0.02,rounding_size=0.05',lw=0,fc=LGREY,transform=a.transAxes))
    a.add_patch(FancyBboxPatch((.04,.08),.03,.84,boxstyle='square,pad=0',lw=0,fc=c,transform=a.transAxes))
    a.text(.13,.62,v,fontsize=22,fontweight='bold',color=c,transform=a.transAxes,va='center')
    a.text(.13,.30,t,fontsize=10,fontweight='bold',color=DARK,transform=a.transAxes,va='center')
    a.text(.13,.16,s,fontsize=8.5,color=GREY,transform=a.transAxes,va='center')
plt.tight_layout(); plt.show()

## 3. Objective 1 & 3 — Where technical issues are misrouted
**Q:** which technical topics leak, and where do they land? Vague intents
(`general_fault`, `general_difficulty`) are expected to misroute most.

In [ ]:
g = q(f'''
SELECT technical_topic_type, COUNT(*) sessions, COUNTIF(is_hard_misroute) hard,
       ROUND(COUNTIF(is_hard_misroute)/COUNT(*)*100,2) pct
FROM {M} WHERE {W} AND is_technical_topic
GROUP BY 1 ORDER BY hard DESC''')
g=g.sort_values('pct')
fig,ax=plt.subplots(figsize=(11,4.2))
cols=[RED if p>=15 else (AMBER if p>=8 else BLUE) for p in g.pct]
b=ax.barh(g.technical_topic_type, g.pct, color=cols)
for bar,n in zip(b,g.hard):
    ax.annotate(f'{bar.get_width():.1f}%  ({int(n):,})',(bar.get_width(),bar.get_y()+bar.get_height()/2),
                xytext=(4,0),textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax.set_title('Hard-misroute rate by technical topic'); ax.set_xlabel('%'); ax.margins(x=0.25)
plt.tight_layout(); plt.show(); g

### Funnel — what happens to a technical request

In [ ]:
f = q(f'''
SELECT COUNTIF(is_technical_topic) tech,
       COUNTIF(is_correct_technical_route) correct,
       COUNTIF(is_technical_topic AND is_bot_contained) contained,
       COUNTIF(is_soft_misroute) soft, COUNTIF(is_hard_misroute) hard
FROM {M} WHERE {W}''').iloc[0]
stages=[('Technical requests',f.tech,INK),('Correctly handled',f.correct,GREEN),
        ('Bot-contained (FCR)',f.contained,BLUE),('Soft misroute',f.soft,AMBER),
        ('Hard misroute',f.hard,RED)]
fig,ax=plt.subplots(figsize=(11,3.8))
for i,(lab,val,c) in enumerate(stages):
    ax.barh(i,val,color=c,height=.62)
    ax.annotate(f'{int(val):,} ({100*val/f.tech:.0f}%)',(val,i),xytext=(6,0),
                textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax.set_yticks(range(len(stages))); ax.set_yticklabels([s[0] for s in stages]); ax.invert_yaxis()
ax.set_title('Technical-request funnel'); ax.margins(x=0.2); plt.tight_layout(); plt.show()

## 4. Objective 2 — Impact (channel load & repeats, not duration)
**Q:** what does misrouting cost? The signal is **handovers and repeat contacts**;
misrouted sessions are not longer.

In [ ]:
c = q(f'''
SELECT CASE WHEN is_hard_misroute THEN 'misrouted_hard'
            WHEN is_soft_misroute THEN 'misrouted_soft'
            WHEN is_correct_technical_route THEN 'correct_technical'
            WHEN is_technical_topic THEN 'technical_other'
            ELSE 'non_technical' END cohort,
       COUNT(*) sessions, ROUND(AVG(n_transfers),2) avg_transfers,
       ROUND(AVG(CAST(repeat_contact_24h AS INT64))*100,1) pct_repeat
FROM {M} WHERE {W} GROUP BY 1''')
order=['non_technical','technical_other','correct_technical','misrouted_soft','misrouted_hard']
c=c.set_index('cohort').reindex(order).dropna(how='all')
fig,ax=plt.subplots(1,2,figsize=(14,4.2))
b1=ax[0].bar(c.index,c.avg_transfers,color=[GREY,BLUE,GREEN,AMBER,RED][:len(c)]); vlabels(ax[0],b1,'{:.2f}')
ax[0].set_title('Avg transfers / session'); ax[0].tick_params(axis='x',rotation=25,labelsize=8)
b2=ax[1].bar(c.index,c.pct_repeat,color=[GREY,BLUE,GREEN,AMBER,RED][:len(c)]); vlabels(ax[1],b2,'{:.0f}%')
ax[1].set_title('Repeat contact within 24h (%)'); ax[1].tick_params(axis='x',rotation=25,labelsize=8)
plt.tight_layout(); plt.show(); c

## 5. Objective 3 — Entry-point driver (channel)
**Q:** which channels misroute most? `voice` is high volume × high rate.
(Note: `CONFIDENCE_LEVEL` is ~all 0/blank in this data, so it is not a usable driver.)

In [ ]:
ch = q(f'''
SELECT CHANNEL, COUNTIF(is_technical_topic) tech, COUNTIF(is_hard_misroute) hard,
       ROUND(COUNTIF(is_hard_misroute)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) pct
FROM {M} WHERE {W} GROUP BY CHANNEL HAVING tech>=5000 ORDER BY pct DESC''')
fig,ax=plt.subplots(figsize=(11,4))
b=ax.bar(ch.CHANNEL, ch.pct, color=INK); vlabels(ax,b,'{:.1f}%')
ax.set_title('Misroute % by channel (>=5k technical sessions)')
ax.tick_params(axis='x',rotation=25); plt.tight_layout(); plt.show(); ch

## 6. Objective 4 — Recommendations: top misroute leaks
**Q:** where do we act first? Each row = a topic → wrong-destination leak,
ranked by an impact score (volume + 2×repeats + handovers).

In [ ]:
leaks = q(f'''
SELECT technical_topic_type, final_transfer_target, routed_support_type,
       COUNT(*) misrouted, COUNTIF(repeat_contact_24h) repeats, SUM(n_transfers) handovers,
       COUNT(*) + 2*COUNTIF(repeat_contact_24h) + SUM(n_transfers) AS impact_score
FROM {M} WHERE {W} AND is_hard_misroute
GROUP BY 1,2,3 ORDER BY impact_score DESC LIMIT 15''')
top=leaks.head(10).iloc[::-1]
fig,ax=plt.subplots(figsize=(12,4.6))
lab=top.technical_topic_type+' -> '+top.final_transfer_target
b=ax.barh(lab, top.misrouted, color=RED); vlabels(ax,b,'{:,.0f}',horiz=True)
ax.set_title('Top 10 misroute leaks'); ax.set_xlabel('misrouted sessions'); ax.margins(x=0.2)
ax.tick_params(axis='y',labelsize=8); plt.tight_layout(); plt.show()
leaks

## 7. Objective 5 — Implementation tracking (FCR & misroute over time)
**Q:** are we improving? Re-run weekly after shipping fixes.

In [ ]:
t = q(f'''
SELECT DATE(START_MOMENT) day,
       ROUND(COUNTIF(is_technical_topic AND is_bot_contained)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) fcr_pct,
       ROUND(COUNTIF(is_hard_misroute)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) misroute_pct
FROM {M} WHERE {W} GROUP BY day ORDER BY day''')
t['day']=pd.to_datetime(t.day)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(t.day, t.misroute_pct.rolling(7,min_periods=1).mean(), color=RED, lw=2.4, label='Hard-misroute % (7d)')
ax.plot(t.day, t.fcr_pct.rolling(7,min_periods=1).mean(), color=GREEN, lw=2.4, label='FCR % (7d)')
ax.set_title('Technical misroute vs FCR over time'); ax.set_ylabel('%'); ax.legend(frameon=False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

---
*All panels query `session_master` (corrected pipeline; see `docs/tag_mappings.md`).
Technical topic from S_ entity ids; routing from the T_ tag outcome + support type.
Cost driver = handovers + repeats, not session length.*